In [2]:
import pandas as pd
from html.parser import HTMLParser
from urllib.request import Request, urlopen

In [3]:
data=\
    (
        pd.read_csv(
            "cof_e_active_churches.csv"
        )
    )

In [4]:
data

,OBJECTID,Church_Name,Church_Code,Type,Listed_building__Grade_,Not_Original_Building,Church_Heritage_Record,Asset_ID,CCB_Type,Secondary_location,...,British_Sign_Language,Autism_Asperger_Aware,Assistance_dogs_welcomed,Braille,Mental_health_support,Makaton,Last_Modified,Open_Date,Shape__Area,Shape__Length
0,1,Bicknacre: St Andrew,608285.0,Mission Church,Not listed / not applicable,NaN,https://facultyonline.churchofengland.org/CHR/...,2447.0,Church,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.324509,37.709063
1,2,St Mark's Church,608682.0,Church Plant,Not listed / not applicable,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2381.135269,349.415140
2,3,Birkenhead: Priory Chapel Priory Street,609395.0,Mission Church,II*,NaN,https://facultyonline.churchofengland.org/CHR/...,3141.0,Church,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.054611,50.662280
3,4,Barrow-in- Furness: St Aidan,607156.0,Parish Church,Not listed / not applicable,NaN,https://facultyonline.churchofengland.org/CHR/...,2000.0,Church,NaN,...,NaN,NaN,Yes,NaN,NaN,NaN,NaN,NaN,394.224991,92.508205
4,5,Victoria Docks: St Luke's Church & School Buil...,608120.0,Parish Church,Not listed / not applicable,NaN,https://facultyonline.churchofengland.org/CHR/...,2303.0,Church,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1587.029793,202.948013
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15343,15471,Cold Salperton: All Saints,616345.0,2nd Parish Ch in Utd Parish,II*,NaN,https://facultyonline.churchofengland.org/CHR/...,5770.0,Church,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136.904526,65.511409
15344,15472,Stockland Green: St Mark,602102.0,Parish Church,Not listed / not applicable,NaN,https://facultyonline.churchofengland.org/CHR/...,773.0,Church,NaN,...,NaN,NaN,Yes,NaN,NaN,NaN,NaN,NaN,205.758568,73.158779
15345,15473,Grendon Bishop & Wacton: St John the Baptist,618045.0,Church,II,NaN,https://facultyonline.churchofengland.org/CHR/...,6097.0,Church,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,88.999008,53.563476
15346,15474,Freedom Church Mereside,603338.0,Church Plant,Not listed / not applicable,NaN,NaN,NaN,NaN,NaN,...,NaN,Yes,Yes,NaN,NaN,NaN,NaN,NaN,339.065262,74.225784


In [10]:
class AnchorParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.anchor_depth = 0
        self.anchor_href = None
        self.anchor_text = []
        self.anchors = []

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            self.anchor_depth += 1
            if self.anchor_depth == 1:
                self.anchor_href = dict(attrs).get("href")
                self.anchor_text = []

    def handle_data(self, data):
        if self.anchor_depth:
            self.anchor_text.append(data)

    def handle_endtag(self, tag):
        if tag == "a" and self.anchor_depth:
            self.anchor_depth -= 1
            if self.anchor_depth == 0:
                text = " ".join("".join(self.anchor_text).split())
                self.anchors.append((text, self.anchor_href))


def get_website_href(acny_link):
    if pd.isna(acny_link) or not str(acny_link).strip():
        return ""

    url = f"{str(acny_link).rstrip('/')}/get-in-touch/"
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=20) as response:
        html = response.read().decode("utf-8", errors="replace")

    parser = AnchorParser()
    parser.feed(html)
    return next(
        (href for text, href in parser.anchors if text.casefold() == "our website"),
        "",
    )


def populate_websites(data):
    if "website" not in data.columns:
        data["website"] = ""

    failures = []
    for count, (index, row) in enumerate(data.iterrows(), start=1):
        if pd.notna(data.at[index, "website"]) and str(data.at[index, "website"]).strip():
            continue

        acny_link = row.get("ACNY_link")
        if pd.isna(acny_link) or not str(acny_link).strip():
            continue

        touch_url = f"{str(acny_link).rstrip('/')}/get-in-touch/"
        try:
            data.at[index, "website"] = get_website_href(acny_link)
        except Exception as exc:
            data.at[index, "website"] = ""
            failures.append((index, touch_url, str(exc)))

        if count % 100 == 0:
            print(f"Processed {count:,} of {len(data):,} churches")

    return failures

In [13]:
website_fetch_errors = populate_websites(data)
print(f"Websites found: {data['website'].astype(bool).sum():,} / {len(data):,}")
print(f"Fetch failures: {len(website_fetch_errors):,}")

Processed 100 of 15,348 churches
Processed 200 of 15,348 churches
Processed 300 of 15,348 churches
Processed 400 of 15,348 churches
Processed 500 of 15,348 churches
Processed 600 of 15,348 churches
Processed 700 of 15,348 churches
Processed 800 of 15,348 churches
Processed 900 of 15,348 churches
Processed 1,000 of 15,348 churches
Processed 1,100 of 15,348 churches
Processed 1,200 of 15,348 churches
Processed 1,300 of 15,348 churches
Processed 1,400 of 15,348 churches
Processed 1,500 of 15,348 churches
Processed 1,600 of 15,348 churches
Processed 1,700 of 15,348 churches
Processed 1,800 of 15,348 churches
Processed 1,900 of 15,348 churches
Processed 2,000 of 15,348 churches
Processed 2,100 of 15,348 churches
Processed 2,200 of 15,348 churches
Processed 2,300 of 15,348 churches
Processed 2,400 of 15,348 churches
Processed 2,500 of 15,348 churches
Processed 2,600 of 15,348 churches
Processed 2,700 of 15,348 churches
Processed 2,800 of 15,348 churches
Processed 2,900 of 15,348 churches
Pro

In [12]:
data

,OBJECTID,Church_Name,Church_Code,Type,Listed_building__Grade_,Not_Original_Building,Church_Heritage_Record,Asset_ID,CCB_Type,Secondary_location,...,Autism_Asperger_Aware,Assistance_dogs_welcomed,Braille,Mental_health_support,Makaton,Last_Modified,Open_Date,Shape__Area,Shape__Length,website
0,1,Bicknacre: St Andrew,608285.0,Mission Church,Not listed / not applicable,NaN,https://facultyonline.churchofengland.org/CHR/...,2447.0,Church,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.324509,37.709063,https://www.wfandbchurch.org
1,2,St Mark's Church,608682.0,Church Plant,Not listed / not applicable,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2381.135269,349.415140,http://www.stmarkscolchester.org.uk
2,3,Birkenhead: Priory Chapel Priory Street,609395.0,Mission Church,II*,NaN,https://facultyonline.churchofengland.org/CHR/...,3141.0,Church,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.054611,50.662280,
3,4,Barrow-in- Furness: St Aidan,607156.0,Parish Church,Not listed / not applicable,NaN,https://facultyonline.churchofengland.org/CHR/...,2000.0,Church,NaN,...,NaN,Yes,NaN,NaN,NaN,NaN,NaN,394.224991,92.508205,https://www.facebook.com/staidansbarrow/
4,5,Victoria Docks: St Luke's Church & School Buil...,608120.0,Parish Church,Not listed / not applicable,NaN,https://facultyonline.churchofengland.org/CHR/...,2303.0,Church,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1587.029793,202.948013,http://www.stlukese16.co.uk/
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15343,15471,Cold Salperton: All Saints,616345.0,2nd Parish Ch in Utd Parish,II*,NaN,https://facultyonline.churchofengland.org/CHR/...,5770.0,Church,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136.904526,65.511409,
15344,15472,Stockland Green: St Mark,602102.0,Parish Church,Not listed / not applicable,NaN,https://facultyonline.churchofengland.org/CHR/...,773.0,Church,NaN,...,NaN,Yes,NaN,NaN,NaN,NaN,NaN,205.758568,73.158779,
15345,15473,Grendon Bishop & Wacton: St John the Baptist,618045.0,Church,II,NaN,https://facultyonline.churchofengland.org/CHR/...,6097.0,Church,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,88.999008,53.563476,
15346,15474,Freedom Church Mereside,603338.0,Church Plant,Not listed / not applicable,NaN,NaN,NaN,NaN,NaN,...,Yes,Yes,NaN,NaN,NaN,NaN,NaN,339.065262,74.225784,


In [15]:
website_fetch_errors

[(9,
  'https://www.achurchnearyou.com/church/15419/get-in-touch/',
  'HTTP Error 404: Not Found'),
 (1649,
  'https://www.achurchnearyou.com/church/33219/get-in-touch/',
  'HTTP Error 404: Not Found'),
 (1704,
  'https://www.achurchnearyou.com/church/21720/get-in-touch/',
  'HTTP Error 404: Not Found'),
 (1793,
  'https://www.achurchnearyou.com/church/21717/get-in-touch/',
  'HTTP Error 404: Not Found'),
 (1809,
  'https://www.achurchnearyou.com/church/4675/get-in-touch/',
  'HTTP Error 404: Not Found'),
 (1988,
  'https://www.achurchnearyou.com/church/33210/get-in-touch/',
  'HTTP Error 404: Not Found'),
 (2821,
  'https://www.achurchnearyou.com/church/18839/get-in-touch/',
  'HTTP Error 404: Not Found'),
 (8436,
  'https://www.achurchnearyou.com/church/15945/get-in-touch/',
  'HTTP Error 404: Not Found'),
 (9092,
  'https://www.achurchnearyou.com/church/3523/get-in-touch/',
  'HTTP Error 404: Not Found'),
 (9511,
  'https://www.achurchnearyou.com/church/15315/get-in-touch/',
  'The 

In [14]:
(
    data
    .to_csv("cof_e_active_churches_with_websites.csv", index=False, encoding="utf-8")
)

In [ ]:
#Run the cny_fetch.py script to fetch the sort codes for the churches with websites.

In [3]:
#check the output as it runs
(
    pd
    .read_csv("scraped_sort_codes2.csv")
    .query("source_url.notna()")
    .query("source_url.str.count('/') > 2")
)

,Asset_ID,cleansed_base_url,sort_code,source_url
13,2303.0,http://www.stlukese16.co.uk,09-01-53,https://www.stlukese16.co.uk/giving
24,1673.0,http://www.cornilochurches.org.uk,52-30-23,https://www.cornilochurches.org.uk/donations.htm
37,1674.0,http://www.cornilochurches.org.uk,52-30-23,https://www.cornilochurches.org.uk/donations.htm
63,1688.0,https://www.margate.church,40-17-34,https://www.margate.church/give
64,1688.0,https://www.margate.church,30-95-45,https://www.margate.church/give
...,...,...,...,...
11067,442.0,https://www.stvigorstjohn.church/contact-us/ho...,60-23-06,https://www.stvigorstjohn.church/church/giving...
11068,1408.0,http://almondsburychurch.org.uk,40-14-20,https://almondsburychurch.org.uk/giving
11071,5093.0,https://www.bay.church,56-00-49,https://www.bay.church/give
11075,460.0,http://www.christchurch-clevedon.org.uk,40-17-50,https://www.christchurch-clevedon.org.uk/giving
